In [1]:
! pip install langchain
! pip install langchain-neo4j langchain-groq

In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain

load_dotenv()

# 1. Configurar a Chave da API da Groq

# 2. Conectar o LangChain ao seu Neo4j local
graph = Neo4jGraph(
    url="bolt://localhost:7687",
    username=os.getenv("NEO4J_USERNAME"),
    password=os.getenv("NEO4J_PASSWORD")
)

# 3. Inicializar o "Cérebro" (Usando o Llama 3 de 70 bilhões de parâmetros)
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0 
)

# 4. Criar o Agente Text-to-Cypher
chain = GraphCypherQAChain.from_llm(
    graph=graph,
    llm=llm,
    verbose=True, 
    allow_dangerous_requests=True 
)

# 5. O Teste de Fogo
pergunta = "Qual foi o valor total que o cliente gastou com Uber?"
print(f"Usuário: {pergunta}\n")

# Invocando o Agente
resposta = chain.invoke({"query": pergunta})

print(f"\nResposta da IA: {resposta['result']}")

👤 Usuário: Qual foi o valor total que o cliente gastou com Uber?



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Cliente)-[:REALIZOU]->(t:Transacao)-[:NO_ESTABELECIMENTO]->(e:Estabelecimento)
WHERE e.nome = 'Uber'
RETURN sum(t.valor) AS total_gasto;
Full Context:
[{'total_gasto': 132.3}]

> Finished chain.

🤖 Resposta da IA: O cliente gastou um total de 132,3.
